# EMIT L1B radiance download

Download EMIT L1B radiance granules with and without a plume for three sites using NASA `earthaccess`. Each granule contains the radiance (`RAD`) and observation geometry (`OBS`) NetCDF files.

In [1]:
from pathlib import Path

import earthaccess
import xarray as xr

DATA_DIR = Path(".")

SCENES = {
    "turkmenistan": {
        "plume": "EMIT_L1B_RAD_001_20260615T081124_2616605_031",
        "no_plume": "EMIT_L1B_RAD_001_20230729T100618_2321007_050",
    },
    "us": {
        "plume": "EMIT_L1B_RAD_001_20260819T180247_2623112_032",
        "no_plume": "EMIT_L1B_RAD_001_20260815T193735_2622713_031",
    },
    "chile": {
        "plume": "EMIT_L1B_RAD_001_20250515T181208_2513512_003",
        "no_plume": "EMIT_L1B_RAD_001_20260115T171148_2601511_005",
    },
}

## Login

Uses Earthdata credentials from environment variables, `~/.netrc`, or an interactive prompt. Pass `persist=True` to save them to `~/.netrc`.

In [3]:
auth = earthaccess.login(strategy="all")
assert auth.authenticated

## Search

In [4]:
granules = {}
for site, names in SCENES.items():
    results = earthaccess.search_data(
        short_name="EMITL1BRAD", version="001", granule_name=list(names.values())
    )
    assert len(results) == len(names), f"{site}: found {len(results)} granules"
    granules[site] = results
    for granule in results:
        print(f"{site:13s} {granule['umm']['GranuleUR']}  {granule.size:.0f} MB")

turkmenistan  EMIT_L1B_RAD_001_20230729T100618_2321007_050  1873 MB
turkmenistan  EMIT_L1B_RAD_001_20260615T081124_2616605_031  1873 MB
us            EMIT_L1B_RAD_001_20260815T193735_2622713_031  1873 MB
us            EMIT_L1B_RAD_001_20260819T180247_2623112_032  1873 MB
chile         EMIT_L1B_RAD_001_20250515T181208_2513512_003  1873 MB
chile         EMIT_L1B_RAD_001_20260115T171148_2601511_005  1873 MB


## Download

In [5]:
files = {}
for site, results in granules.items():
    files[site] = [
        Path(p) for p in earthaccess.download(results, local_path=DATA_DIR / site)
    ]

files

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

{'turkmenistan': [PosixPath('turkmenistan/EMIT_L1B_RAD_001_20230729T100618_2321007_050.nc'),
  PosixPath('turkmenistan/EMIT_L1B_OBS_001_20230729T100618_2321007_050.nc'),
  PosixPath('turkmenistan/EMIT_L1B_RAD_001_20260615T081124_2616605_031.nc'),
  PosixPath('turkmenistan/EMIT_L1B_OBS_001_20260615T081124_2616605_031.nc')],
 'us': [PosixPath('us/EMIT_L1B_RAD_001_20260815T193735_2622713_031.nc'),
  PosixPath('us/EMIT_L1B_OBS_001_20260815T193735_2622713_031.nc'),
  PosixPath('us/EMIT_L1B_RAD_001_20260819T180247_2623112_032.nc'),
  PosixPath('us/EMIT_L1B_OBS_001_20260819T180247_2623112_032.nc')],
 'chile': [PosixPath('chile/EMIT_L1B_RAD_001_20250515T181208_2513512_003.nc'),
  PosixPath('chile/EMIT_L1B_OBS_001_20250515T181208_2513512_003.nc'),
  PosixPath('chile/EMIT_L1B_RAD_001_20260115T171148_2601511_005.nc'),
  PosixPath('chile/EMIT_L1B_OBS_001_20260115T171148_2601511_005.nc')]}

## Quick check

In [6]:
rad_path = DATA_DIR / "turkmenistan" / f"{SCENES['turkmenistan']['plume']}.nc"
xr.open_dataset(rad_path)

<xarray.Dataset> Size: 2GB
Dimensions:            (downtrack: 1280, crosstrack: 1242, bands: 285)
Dimensions without coordinates: downtrack, crosstrack, bands
Data variables:
    radiance           (downtrack, crosstrack, bands) float32 2GB ...
    flat_field_update  (crosstrack, bands) float32 1MB ...
Attributes: (12/38)
    ncei_template_version:             NCEI_NetCDF_Swath_Template_v2.0
    summary:                           The Earth Surface Mineral Dust Source ...
    keywords:                          Imaging Spectroscopy, minerals, EMIT, ...
    sensor:                            EMIT (Earth Surface Mineral Dust Sourc...
    instrument:                        EMIT
    platform:                          ISS
    ...                                ...
    southernmost_latitude:             39.566383248989425
    spatialResolution:                 0.000542232520256367
    spatial_ref:                       GEOGCS["WGS 84",DATUM["WGS_1984",SPHER...
    geotransform:                      [ 6.06169817e+01  5.42232520e-04 -0.00...
    day_night_flag:                    Day
    title:                             EMIT L1B At-Sensor Calibrated Radiance...